# Source Injection into Rubin Images

Uses the official `lsst.source.injection` tools — same as the DP0.2 tutorial:
[DP02_14](https://github.com/rubin-dp0/tutorial-notebooks/blob/main/DP02_14_Injecting_Synthetic_Sources.ipynb)

**Contents:**
1. Inject into a **calexp** (visit-level image)
2. Presentation plots: before/after/diff, postage stamps, diagnostics
3. Inject into a **deepCoadd** (coadded image)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle, FancyArrowPatch
import matplotlib.gridspec as gridspec
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack

from lsst.daf.butler import Butler
import lsst.afw.display as afwDisplay
import lsst.geom as geom
from lsst.source.injection import generate_injection_catalog
from lsst.source.injection import VisitInjectConfig, VisitInjectTask

afwDisplay.setDefaultBackend('matplotlib')
plt.style.use('tableau-colorblind10')
print('Imports OK')

---
# Part A: Calexp Injection

## 1. Load a calexp

In [ ]:
butler = Butler('dp02', collections='2.2i/runs/DP0.2')
print('Butler connected')

In [ ]:
tract = 3828
where = f"instrument='LSSTCam-imSim' AND skymap='DC2' AND tract={tract} AND detector=19 AND band='i'"
calexp_refs = sorted(list(set(butler.query_datasets('calexp', where=where))))
print(f'Found {len(calexp_refs)} calexps')

dataId = calexp_refs[5].dataId
print(f'Using: {dataId}')

In [ ]:
calexp = butler.get('calexp', dataId=dataId)

wcs = calexp.getWcs()
bbox = calexp.getBBox()
psf = calexp.getPsf()
photo_calib = calexp.getPhotoCalib()

boxcen = bbox.getCenter()
cen = wcs.pixelToSky(boxcen)
sc_cen = SkyCoord(ra=cen[0].asDegrees()*u.deg, dec=cen[1].asDegrees()*u.deg)

imsize = bbox.getDimensions()[0] * wcs.getPixelScale().asDegrees()
inject_size = imsize / 2.0
pixel_scale = wcs.getPixelScale().asArcseconds()

print(f'Image       : {bbox}')
print(f'Center      : {sc_cen}')
print(f'Size on sky : {imsize:.4f} deg')
print(f'Pixel scale : {pixel_scale:.4f} arcsec/px')
print(f'PSF type    : {type(psf).__name__}')

## 2. Build injection catalog

In [ ]:
ra_lo, ra_hi = sc_cen.ra.value - inject_size, sc_cen.ra.value + inject_size
dec_lo, dec_hi = sc_cen.dec.value - inject_size, sc_cen.dec.value + inject_size

# Galaxies — Sersic profiles
cat_galaxies = generate_injection_catalog(
    ra_lim=[ra_lo, ra_hi], dec_lim=[dec_lo, dec_hi],
    number=3, seed='1234', source_type='Sersic',
    mag=[18.0, 20.0, 22.0],
    n=[1, 2, 4], q=[0.9, 0.5], beta=[0.0, 90.0],
    half_light_radius=[10.0, 20.0],
)

# Stars — point sources
cat_stars = generate_injection_catalog(
    ra_lim=[ra_lo, ra_hi], dec_lim=[dec_lo, dec_hi],
    number=10, seed='5678', source_type='Star',
    mag=[17.0, 18.0, 19.0, 20.0, 21.0],
)

inject_cat = vstack([cat_galaxies, cat_stars])
print(f'Galaxies: {len(cat_galaxies)}, Stars: {len(cat_stars)}, Total: {len(inject_cat)}')

## 3. Inject sources

In [ ]:
inject_config = VisitInjectConfig()
inject_task = VisitInjectTask(config=inject_config)

injected_output = inject_task.run(
    injection_catalogs=inject_cat,
    input_exposure=calexp.clone(),
    psf=psf,
    photo_calib=photo_calib,
    wcs=wcs,
)

injected_exposure = injected_output.output_exposure
injected_catalog = injected_output.output_catalog

print(f'Injected {len(injected_catalog)} sources')

# Extract arrays for plotting
orig_arr = calexp.image.array.astype(float)
inj_arr  = injected_exposure.image.array.astype(float)
diff_arr = inj_arr - orig_arr
ny, nx   = orig_arr.shape

# Injected source positions
has_xy = ('x' in injected_catalog.colnames and 'y' in injected_catalog.colnames)
if has_xy:
    inj_x = np.array(injected_catalog['x'])
    inj_y = np.array(injected_catalog['y'])
    inj_mag = np.array(injected_catalog['mag'])
    inj_type = np.array(injected_catalog['source_type'])
    print(f'  Magnitude range: [{inj_mag.min():.1f}, {inj_mag.max():.1f}]')
    print(f'  Source types: {np.unique(inj_type)}')

---
## 4. Presentation Plots

### 4.1 Before / After / Difference (full image)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
v1, v2 = np.percentile(orig_arr, [1, 99])

axes[0].imshow(orig_arr, cmap='gray', origin='lower', vmin=v1, vmax=v2)
axes[0].set_title('(a) Original calexp', fontsize=13)

axes[1].imshow(inj_arr, cmap='gray', origin='lower', vmin=v1, vmax=v2)
if has_xy:
    for i in range(len(inj_x)):
        c = 'cyan' if inj_type[i] == 'Star' else 'lime'
        axes[1].plot(inj_x[i], inj_y[i], 'o', ms=8, mfc='none', mec=c, mew=1.3)
    axes[1].plot([], [], 'o', mfc='none', mec='cyan', ms=8, label='Star')
    axes[1].plot([], [], 'o', mfc='none', mec='lime', ms=8, label='Sersic galaxy')
    axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_title(f'(b) After injection ({len(injected_catalog)} sources)', fontsize=13)

d_show = diff_arr.copy()
d_show[d_show <= 0] = np.nan
vd = d_show[np.isfinite(d_show)]
if len(vd) > 0:
    im = axes[2].imshow(d_show, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=max(0.1, np.percentile(vd, 1)), vmax=np.nanmax(vd)))
    plt.colorbar(im, ax=axes[2], shrink=0.85, label='Injected flux (counts)')
axes[2].set_title('(c) Difference (injected signal only)', fontsize=13)

for ax in axes:
    ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')

fig.suptitle(f'Source Injection into Calexp — i-band, tract {tract}', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('plot_calexp_before_after_diff.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Postage Stamps — Before / After / Injected Only

In [ ]:
if has_xy:
    # Sort by magnitude (bright first)
    sort_idx = np.argsort(inj_mag)
    n_show = min(10, len(sort_idx))
    pick = sort_idx[np.linspace(0, len(sort_idx)-1, n_show, dtype=int)]

    fig, axes = plt.subplots(3, n_show, figsize=(2.4*n_show, 7))
    for col_i, idx in enumerate(pick):
        cx, cy = int(inj_x[idx]), int(inj_y[idx])
        r = 60
        y0, y1 = max(0, cy-r), min(ny, cy+r)
        x0, x1 = max(0, cx-r), min(nx, cx+r)

        s_orig = orig_arr[y0:y1, x0:x1]
        s_inj  = inj_arr[y0:y1, x0:x1]
        s_diff = diff_arr[y0:y1, x0:x1]
        vp1, vp2 = np.percentile(s_inj, [0.5, 99.5])

        axes[0, col_i].imshow(s_orig, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
        axes[0, col_i].set_title(f'mag={inj_mag[idx]:.1f}', fontsize=8)

        axes[1, col_i].imshow(s_inj, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
        axes[1, col_i].set_title(f'{inj_type[idx]}', fontsize=8)

        dp = np.clip(s_diff, 0, None)
        if dp.max() > 0:
            axes[2, col_i].imshow(dp, cmap='inferno', origin='lower',
                                  norm=LogNorm(vmin=max(dp[dp>0].min(), 0.01), vmax=dp.max()))

        for row in range(3):
            axes[row, col_i].plot(cx-x0, cy-y0, 'r+', ms=7, mew=1)
            axes[row, col_i].tick_params(labelsize=5)
            axes[row, col_i].set_xticks([]); axes[row, col_i].set_yticks([])

    axes[0, 0].set_ylabel('Before', fontsize=11)
    axes[1, 0].set_ylabel('After', fontsize=11)
    axes[2, 0].set_ylabel('Injected only', fontsize=11)
    fig.suptitle('Postage Stamps of Injected Sources (sorted by magnitude)', fontsize=13)
    plt.tight_layout()
    plt.savefig('plot_calexp_postage_stamps.png', dpi=150, bbox_inches='tight')
    plt.show()

---
# Part B: deepCoadd Injection

The `source_injection` package also supports coadd-level injection
via `CoaddInjectTask`. This handles the PSF internally through GalSim —
**no manual PSF extraction needed**.

For coadds, the `CoaddInjectTask`:
- Uses the exposure's WCS to place sources at RA/Dec positions
- Uses the exposure's PhotoCalib to convert magnitudes to flux
- Convolves injected sources with the coadd PSF model internally via GalSim

Reference: [source_injection docs](https://pipelines.lsst.io/v/daily/modules/lsst.source.injection/index.html)

In [ ]:
from lsst.source.injection import CoaddInjectConfig, CoaddInjectTask
print('CoaddInject imports OK')

In [ ]:
# Load a deepCoadd
coadd_dataId = {'tract': 3828, 'patch': 24, 'band': 'i'}

deepCoadd = butler.get('deepCoadd', dataId=coadd_dataId)

coadd_wcs = deepCoadd.getWcs()
coadd_bbox = deepCoadd.getBBox()
coadd_psf = deepCoadd.getPsf()
coadd_photo_calib = deepCoadd.getPhotoCalib()

coadd_boxcen = coadd_bbox.getCenter()
coadd_cen = coadd_wcs.pixelToSky(coadd_boxcen)
coadd_sc_cen = SkyCoord(ra=coadd_cen[0].asDegrees()*u.deg, dec=coadd_cen[1].asDegrees()*u.deg)
coadd_imsize = coadd_bbox.getDimensions()[0] * coadd_wcs.getPixelScale().asDegrees()
coadd_inject_size = coadd_imsize / 2.0

print(f'deepCoadd    : {coadd_dataId}')
print(f'Shape        : {deepCoadd.image.array.shape}')
print(f'BBox         : {coadd_bbox}')
print(f'Center       : {coadd_sc_cen}')
print(f'PSF type     : {type(coadd_psf).__name__}')

In [ ]:
# Build injection catalog for the coadd
coadd_ra_lo = coadd_sc_cen.ra.value - coadd_inject_size
coadd_ra_hi = coadd_sc_cen.ra.value + coadd_inject_size
coadd_dec_lo = coadd_sc_cen.dec.value - coadd_inject_size
coadd_dec_hi = coadd_sc_cen.dec.value + coadd_inject_size

coadd_cat_gal = generate_injection_catalog(
    ra_lim=[coadd_ra_lo, coadd_ra_hi],
    dec_lim=[coadd_dec_lo, coadd_dec_hi],
    number=5, seed='9876', source_type='Sersic',
    mag=[18.0, 20.0, 22.0, 24.0],
    n=[1, 2, 4], q=[0.8, 0.4], beta=[30.0, 120.0],
    half_light_radius=[5.0, 15.0, 30.0],
)

coadd_cat_stars = generate_injection_catalog(
    ra_lim=[coadd_ra_lo, coadd_ra_hi],
    dec_lim=[coadd_dec_lo, coadd_dec_hi],
    number=15, seed='1111', source_type='Star',
    mag=[17.0, 18.0, 19.0, 20.0, 21.0, 22.0],
)

coadd_inject_cat = vstack([coadd_cat_gal, coadd_cat_stars])
print(f'Coadd catalog: {len(coadd_cat_gal)} galaxies + {len(coadd_cat_stars)} stars = {len(coadd_inject_cat)} total')

In [ ]:
# Inject using CoaddInjectTask — PSF handled internally via GalSim
coadd_inject_config = CoaddInjectConfig()
coadd_inject_task = CoaddInjectTask(config=coadd_inject_config)

coadd_injected_output = coadd_inject_task.run(
    injection_catalogs=coadd_inject_cat,
    input_exposure=deepCoadd.clone(),
    psf=coadd_psf,
    photo_calib=coadd_photo_calib,
    wcs=coadd_wcs,
)

coadd_injected_exp = coadd_injected_output.output_exposure
coadd_injected_cat = coadd_injected_output.output_catalog

print(f'Injected {len(coadd_injected_cat)} sources into deepCoadd')

In [ ]:
# Extract arrays
coadd_orig_arr = deepCoadd.image.array.astype(float)
coadd_inj_arr  = coadd_injected_exp.image.array.astype(float)
coadd_diff_arr = coadd_inj_arr - coadd_orig_arr
coadd_ny, coadd_nx = coadd_orig_arr.shape

coadd_has_xy = ('x' in coadd_injected_cat.colnames and 'y' in coadd_injected_cat.colnames)
if coadd_has_xy:
    coadd_inj_x = np.array(coadd_injected_cat['x'])
    coadd_inj_y = np.array(coadd_injected_cat['y'])
    coadd_inj_mag = np.array(coadd_injected_cat['mag'])
    coadd_inj_type = np.array(coadd_injected_cat['source_type'])

### deepCoadd: Before / After / Difference

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
cv1, cv2 = np.percentile(coadd_orig_arr, [1, 99])

axes[0].imshow(coadd_orig_arr, cmap='gray', origin='lower', vmin=cv1, vmax=cv2)
axes[0].set_title('(a) Original deepCoadd', fontsize=13)

axes[1].imshow(coadd_inj_arr, cmap='gray', origin='lower', vmin=cv1, vmax=cv2)
if coadd_has_xy:
    for i in range(len(coadd_inj_x)):
        c = 'cyan' if coadd_inj_type[i] == 'Star' else 'lime'
        axes[1].plot(coadd_inj_x[i], coadd_inj_y[i], 'o', ms=8, mfc='none', mec=c, mew=1.3)
    axes[1].plot([], [], 'o', mfc='none', mec='cyan', ms=8, label='Star')
    axes[1].plot([], [], 'o', mfc='none', mec='lime', ms=8, label='Sersic')
    axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_title(f'(b) After injection ({len(coadd_injected_cat)} sources)', fontsize=13)

cd_show = coadd_diff_arr.copy()
cd_show[cd_show <= 0] = np.nan
cd_valid = cd_show[np.isfinite(cd_show)]
if len(cd_valid) > 0:
    im = axes[2].imshow(cd_show, cmap='hot', origin='lower',
                        norm=LogNorm(vmin=max(0.1, np.percentile(cd_valid, 1)), vmax=np.nanmax(cd_valid)))
    plt.colorbar(im, ax=axes[2], shrink=0.85, label='Injected flux')
axes[2].set_title('(c) Difference (injected signal only)', fontsize=13)

for ax in axes:
    ax.set_xlabel('X (pixels)'); ax.set_ylabel('Y (pixels)')
fig.suptitle(f'deepCoadd Injection — {coadd_dataId}', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('plot_coadd_before_after_diff.png', dpi=150, bbox_inches='tight')
plt.show()

### deepCoadd: Postage Stamps

In [ ]:
if coadd_has_xy:
    sort_idx = np.argsort(coadd_inj_mag)
    n_show = min(10, len(sort_idx))
    pick = sort_idx[np.linspace(0, len(sort_idx)-1, n_show, dtype=int)]

    fig, axes = plt.subplots(3, n_show, figsize=(2.4*n_show, 7))
    for col_i, idx in enumerate(pick):
        cx, cy = int(coadd_inj_x[idx]), int(coadd_inj_y[idx])
        r = 60
        y0, y1 = max(0, cy-r), min(coadd_ny, cy+r)
        x0, x1 = max(0, cx-r), min(coadd_nx, cx+r)

        s_orig = coadd_orig_arr[y0:y1, x0:x1]
        s_inj  = coadd_inj_arr[y0:y1, x0:x1]
        s_diff = coadd_diff_arr[y0:y1, x0:x1]
        vp1, vp2 = np.percentile(s_inj, [0.5, 99.5])

        axes[0, col_i].imshow(s_orig, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
        axes[0, col_i].set_title(f'mag={coadd_inj_mag[idx]:.1f}', fontsize=8)
        axes[1, col_i].imshow(s_inj, cmap='gray', origin='lower', vmin=vp1, vmax=vp2)
        axes[1, col_i].set_title(f'{coadd_inj_type[idx]}', fontsize=8)
        dp = np.clip(s_diff, 0, None)
        if dp.max() > 0:
            axes[2, col_i].imshow(dp, cmap='inferno', origin='lower',
                                  norm=LogNorm(vmin=max(dp[dp>0].min(), 0.01), vmax=dp.max()))
        for row in range(3):
            axes[row, col_i].plot(cx-x0, cy-y0, 'r+', ms=7, mew=1)
            axes[row, col_i].set_xticks([]); axes[row, col_i].set_yticks([])

    axes[0, 0].set_ylabel('Before', fontsize=11)
    axes[1, 0].set_ylabel('After', fontsize=11)
    axes[2, 0].set_ylabel('Injected only', fontsize=11)
    fig.suptitle('deepCoadd Postage Stamps (sorted by magnitude)', fontsize=13)
    plt.tight_layout()
    plt.savefig('plot_coadd_postage_stamps.png', dpi=150, bbox_inches='tight')
    plt.show()

### deepCoadd: Summary

In [ ]:
if coadd_has_xy:
    coadd_bg_std = np.std(coadd_orig_arr)
    coadd_peak_sn = []
    for i in range(len(coadd_inj_x)):
        cx, cy = int(coadd_inj_x[i]), int(coadd_inj_y[i])
        r = 30
        y0, y1 = max(0, cy-r), min(coadd_ny, cy+r)
        x0, x1 = max(0, cx-r), min(coadd_nx, cx+r)
        d = coadd_diff_arr[y0:y1, x0:x1]
        coadd_peak_sn.append(d.max() / coadd_bg_std if coadd_bg_std > 0 else 0)
    coadd_peak_sn = np.array(coadd_peak_sn)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Mag histogram
    sm = coadd_inj_type == 'Star'
    axes[0].hist(coadd_inj_mag[sm], bins=10, alpha=0.7, color='steelblue', edgecolor='black', label='Stars')
    axes[0].hist(coadd_inj_mag[~sm], bins=10, alpha=0.7, color='darkorange', edgecolor='black', label='Galaxies')
    axes[0].set_xlabel('Magnitude'); axes[0].set_ylabel('Count')
    axes[0].set_title('Magnitude Distribution'); axes[0].legend()

    # Peak S/N
    axes[1].scatter(coadd_inj_mag, coadd_peak_sn,
                    c=['steelblue' if t=='Star' else 'darkorange' for t in coadd_inj_type],
                    s=50, edgecolors='black', lw=0.5)
    axes[1].axhline(5, color='red', ls='--', lw=1, label='S/N=5')
    axes[1].axhline(3, color='orange', ls='--', lw=1, label='S/N=3')
    axes[1].set_xlabel('Magnitude'); axes[1].set_ylabel('Peak S/N')
    axes[1].set_title('Peak S/N vs Magnitude'); axes[1].set_yscale('log')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    # Summary
    axes[2].axis('off')
    txt = (
        f'deepCoadd Injection Summary\n'
        f'───────────────────────────\n'
        f'DataId       : {coadd_dataId}\n'
        f'Image shape  : {coadd_orig_arr.shape}\n'
        f'\n'
        f'N injected   : {len(coadd_injected_cat)}\n'
        f'  Stars      : {int(sm.sum())}\n'
        f'  Galaxies   : {int((~sm).sum())}\n'
        f'Mag range    : [{coadd_inj_mag.min():.1f}, {coadd_inj_mag.max():.1f}]\n'
        f'BG std       : {coadd_bg_std:.2f}\n'
        f'Median S/N   : {np.median(coadd_peak_sn):.1f}\n'
        f'S/N > 5      : {(coadd_peak_sn > 5).sum()}/{len(coadd_peak_sn)}\n'
        f'\n'
        f'Method: CoaddInjectTask\n'
        f'PSF: handled internally by\n'
        f'     GalSim (no manual extraction)'
    )
    axes[2].text(0.05, 0.95, txt, transform=axes[2].transAxes, fontsize=9.5, va='top',
                family='monospace', bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

    fig.suptitle('deepCoadd Injection — Diagnostic Summary', fontsize=14)
    plt.tight_layout()
    plt.savefig('plot_coadd_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Comparison: Calexp vs deepCoadd injection

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Calexp row
cv1c, cv2c = np.percentile(orig_arr, [1, 99])
axes[0, 0].imshow(orig_arr, cmap='gray', origin='lower', vmin=cv1c, vmax=cv2c)
axes[0, 0].set_title('Calexp — Original')
axes[0, 1].imshow(inj_arr, cmap='gray', origin='lower', vmin=cv1c, vmax=cv2c)
axes[0, 1].set_title(f'Calexp — Injected ({len(injected_catalog)})')
d_c = diff_arr.copy(); d_c[d_c <= 0] = np.nan
dv = d_c[np.isfinite(d_c)]
if len(dv) > 0:
    axes[0, 2].imshow(d_c, cmap='hot', origin='lower',
                      norm=LogNorm(vmin=max(0.1, np.percentile(dv, 1)), vmax=np.nanmax(dv)))
axes[0, 2].set_title('Calexp — Difference')

# Coadd row
cv1d, cv2d = np.percentile(coadd_orig_arr, [1, 99])
axes[1, 0].imshow(coadd_orig_arr, cmap='gray', origin='lower', vmin=cv1d, vmax=cv2d)
axes[1, 0].set_title('deepCoadd — Original')
axes[1, 1].imshow(coadd_inj_arr, cmap='gray', origin='lower', vmin=cv1d, vmax=cv2d)
axes[1, 1].set_title(f'deepCoadd — Injected ({len(coadd_injected_cat)})')
d_d = coadd_diff_arr.copy(); d_d[d_d <= 0] = np.nan
ddv = d_d[np.isfinite(d_d)]
if len(ddv) > 0:
    axes[1, 2].imshow(d_d, cmap='hot', origin='lower',
                      norm=LogNorm(vmin=max(0.1, np.percentile(ddv, 1)), vmax=np.nanmax(ddv)))
axes[1, 2].set_title('deepCoadd — Difference')

axes[0, 0].set_ylabel('calexp\n(VisitInjectTask)', fontsize=12)
axes[1, 0].set_ylabel('deepCoadd\n(CoaddInjectTask)', fontsize=12)
for ax in axes.flat:
    ax.set_xlabel('X (pixels)')

fig.suptitle('Comparison: calexp vs deepCoadd Injection', fontsize=15)
plt.tight_layout()
plt.savefig('plot_calexp_vs_coadd.png', dpi=150, bbox_inches='tight')
plt.show()

## Generated Figures

In [ ]:
import os, glob
figs = sorted(glob.glob('plot_*.png'))
print(f'{len(figs)} figures generated:')
for f in figs:
    print(f'  {f:45s} {os.path.getsize(f)/1024:6.0f} KB')

## Summary

| Image Type | Task | PSF Handling |
|------------|------|--------------|
| `calexp` | `VisitInjectTask` | PSF from calexp, convolved by GalSim internally |
| `deepCoadd` | `CoaddInjectTask` | CoaddPsf model, convolved by GalSim internally |

Both tasks use **`lsst.source.injection`** + **GalSim** under the hood.
No manual PSF extraction or scipy convolution needed —
the pipeline handles PSF convolution, WCS, and photometric calibration automatically.